In [8]:
import sys
import os
import numpy as np

sys.path.append(os.path.abspath('..'))

from LSM.stochastic_processes import GeometricBrownianMotion
from LSM.payoffs import AmericanOption
from LSM.regression_bases import LaguerrePolynomials
from LSM.algorithms import LeastSquaresMonteCarlo

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[autoreload of LSM.stochastic_processes failed: Traceback (most recent call last):
  File "c:\Users\ningy\miniconda3\envs\algo_proj\lib\site-packages\IPython\extensions\autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "c:\Users\ningy\miniconda3\envs\algo_proj\lib\site-packages\IPython\extensions\autoreload.py", line 500, in superreload
    update_generic(old_obj, new_obj)
  File "c:\Users\ningy\miniconda3\envs\algo_proj\lib\site-packages\IPython\extensions\autoreload.py", line 397, in update_generic
    update(a, b)
  File "c:\Users\ningy\miniconda3\envs\algo_proj\lib\site-packages\IPython\extensions\autoreload.py", line 349, in update_class
    if update_generic(old_obj, new_obj):
  File "c:\Users\ningy\miniconda3\envs\algo_proj\lib\site-packages\IPython\extensions\autoreload.py", line 397, in update_generic
    update(a, b)
  File "c:\Users\ningy\miniconda3\envs\algo_proj\lib\site-packages\IPython\extensions\autoreload.py", line 309, in update_fu

In [ ]:
# TODO: Sanity check: an American call without dividend (q=0) is equivalent to a 
# European call. Compare our LSM price with the BSM European call price.


 
# TODO: Use the American put numerical example in Longstaff, Schartz (2001) pp. 126-128.
# Goals: 
# a) replicate their results; 
# b) compare with other benchmarks and libraries (e.g. CRR Binomial Tree),
# focusing on accuracy (prices), speed (%timeit) and memory usage
# [CREATE TESTS USING OTHER LIBRARIES FIRST!]


S0 = 36.0
r = 0.06
q = 0.0
sigma = 0.2
K = 40.0
T = 1.0
n_steps = 50
n_paths = 10000


gbm = GeometricBrownianMotion(S0=S0, r=r, q=q, sigma=sigma)
put_payoff = AmericanOption(strike=K, option_type="call") # "put"
laguerre_basis = LaguerrePolynomials(degree=3)


lsm_engine = LeastSquaresMonteCarlo(
    process=gbm, 
    payoff_function=put_payoff, 
    basis_function=laguerre_basis
)


time_grid, paths = gbm.simulate(T=T, n_steps=n_steps, n_paths=n_paths)
print(f"1. GBM paths shape: {paths.shape}  --> Expect ({n_paths}, {n_steps + 1})")

payoff_at_maturity = put_payoff(paths[:, -1])
print(f"2. Payoff shape: {payoff_at_maturity.shape}  --> Expect: ({n_paths},)")

test_X = paths[:100, -1]
test_Y = payoff_at_maturity[:100]
beta = laguerre_basis.fit(X=test_X, Y=test_Y)
print(f"3. beta fitted on Laguerre basis \n{beta}  --> Expect: ")


price = lsm_engine.pricer(T=T, n_steps=n_steps, n_paths=n_paths)
print(f"4. LSM price: {price}  --> Expect: ")

1. GBM paths shape: (10000, 51)  --> Expect (10000, 51)
2. Payoff shape: (10000,)  --> Expect: (10000,)
3. beta fitted on Laguerre basis 
[0. 0. 0. 0.]  --> Expect: 
4. LSM price: 0.0  --> Expect: 
